# UniMol s4 & s5 Training — OpenADMET PXR Phase 2

Trains two diversity UniMol variants (LR=5e-4 and LR=1e-3) on the PXR dataset.

**Before running:**
1. Runtime → Change runtime type → **T4 GPU**
2. Upload `butina_folds.parquet` and `openadmet_test_std.parquet` to Google Drive at:
   `My Drive/openadmet_pxr/data/`
3. Run all cells top-to-bottom

**Outputs** saved to `My Drive/openadmet_pxr/results/`:
- `unimol2_s4/oof_predictions.npy` + `test_predictions.npy`
- `unimol2_s5/oof_predictions.npy` + `test_predictions.npy`

**Runtime:** ~2–3h per variant on T4. Run sequentially (s4 first, then s5).

In [ ]:
# ── Cell 1: Verify GPU ────────────────────────────────────────────────────────
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout)
if 'failed' in result.stderr.lower() or not result.stdout:
    raise RuntimeError('No GPU detected. Go to Runtime → Change runtime type → T4 GPU')

In [ ]:
# ── Cell 2: Install dependencies (~3 min) ────────────────────────────────────
!pip install -q unimol_tools loguru scipy pyarrow
# Colab already has torch with CUDA — no need to reinstall
import torch
print(f'PyTorch {torch.__version__}, CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
# ── Cell 3: Mount Google Drive ────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

import os
DATA_DIR    = '/content/drive/MyDrive/openadmet_pxr/data'
RESULTS_DIR = '/content/drive/MyDrive/openadmet_pxr/results'
os.makedirs(RESULTS_DIR, exist_ok=True)

# Verify data files are present
for f in ['butina_folds.parquet', 'openadmet_test_std.parquet']:
    path = os.path.join(DATA_DIR, f)
    if not os.path.exists(path):
        raise FileNotFoundError(
            f'{f} not found at {path}\n'
            'Upload it to My Drive/openadmet_pxr/data/ first.'
        )
    print(f'✓ {f} ({os.path.getsize(path)//1024} KB)')

In [ ]:
# ── Cell 4: Inline utilities (device + oof) ───────────────────────────────────
import numpy as np
import torch
from loguru import logger

# ---- device helpers ----
def get_unimol_device_params(base_batch_size=16):
    if torch.cuda.is_available():
        return {'use_gpu': 'all', 'use_amp': True, 'batch_size': base_batch_size}
    return {'use_gpu': False, 'use_amp': False, 'batch_size': base_batch_size}

def clear_device_cache():
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

def augment_smiles(smiles_list, n_aug=10):
    try:
        from rdkit import Chem
    except ImportError:
        return smiles_list, list(range(len(smiles_list)))
    augmented, compound_idx = [], []
    for i, smi in enumerate(smiles_list):
        mol = Chem.MolFromSmiles(smi)
        if mol is None:
            augmented.append(smi); compound_idx.append(i); continue
        seen = set()
        for _ in range(n_aug * 3):
            r = Chem.MolToSmiles(mol, doRandom=True)
            if r not in seen:
                seen.add(r); augmented.append(r); compound_idx.append(i)
            if len(seen) >= n_aug: break
        while len(seen) < n_aug:
            augmented.append(smi); compound_idx.append(i); seen.add(smi+str(len(seen)))
    return augmented, compound_idx

# ---- oof metrics ----
def evaluate_oof_simple(y_true, y_oof, folds):
    from scipy.stats import spearmanr
    valid = ~(np.isnan(y_true) | np.isnan(y_oof))
    y, p, f = y_true[valid], y_oof[valid], folds[valid]
    mae  = float(np.mean(np.abs(y - p)))
    dr   = float(np.max(y) - np.min(y)) or 1.0
    rae  = mae / dr
    rae_test = mae / 0.80
    sp   = spearmanr(y, p).statistic
    logger.info(f'OOF MAE={mae:.4f}, RAE_test={rae_test:.4f}, Spearman={sp:.4f} (n={valid.sum()})')
    return {'mae': mae, 'rae': rae, 'rae_test': rae_test, 'spearman': sp}

print('Utilities loaded.')

In [ ]:
# ── Cell 5: Load data ─────────────────────────────────────────────────────────
import pandas as pd
import gc
from pathlib import Path

train_df = pd.read_parquet(os.path.join(DATA_DIR, 'butina_folds.parquet'))
test_std  = pd.read_parquet(os.path.join(DATA_DIR, 'openadmet_test_std.parquet'))

PRIMARY = {'openadmet', 'analog_set1', 'htchem', 'htchem_semi_pure'}
if 'source' in train_df.columns:
    train_df = train_df[train_df['source'].isin(PRIMARY)].reset_index(drop=True)

pec50_col  = 'pec50_median' if 'pec50_median' in train_df.columns else 'pec50'
smiles_col = 'smiles_std'   if 'smiles_std'   in train_df.columns else 'smiles'
test_smi_col = 'smiles_std' if 'smiles_std'   in test_std.columns  else 'smiles'

train_df   = train_df[train_df[pec50_col].notna()].reset_index(drop=True)
y_train    = train_df[pec50_col].values.astype(np.float32)
smiles_train = train_df[smiles_col].tolist()
smiles_test  = test_std[test_smi_col].tolist()
folds        = train_df['fold'].values

logger.info(f'Train: {len(smiles_train)} compounds, Test: {len(smiles_test)} compounds')
logger.info(f'pEC50 range: {y_train.min():.2f}–{y_train.max():.2f}')

In [ ]:
# ── Cell 6: Training helper ───────────────────────────────────────────────────
import json
from unimol_tools import MolPredict, MolTrain

_DEVICE = get_unimol_device_params(base_batch_size=16)

def train_unimol_variant(variant_name, learning_rate, out_dir_drive):
    """Train one UniMol variant, save OOF + test predictions to Drive."""
    out_dir = Path(f'/content/models/{variant_name}')
    out_dir.mkdir(parents=True, exist_ok=True)

    params = {
        'task': 'regression', 'data_type': 'molecule',
        'model_name': 'unimolv1', 'epochs': 25,
        'learning_rate': learning_rate,
        'batch_size': _DEVICE['batch_size'],
        'early_stopping': 5,
        'use_amp': _DEVICE['use_amp'],
        'use_gpu': _DEVICE['use_gpu'],
        'remove_hs': False, 'kfold': 1, 'split': 'random',
        'metrics': 'mae', 'conf_cache_level': 0,
    }

    logger.info(f'\n=== {variant_name} (LR={learning_rate}) ===')
    oof = np.zeros(len(y_train), dtype=np.float32)

    for fold_id in sorted(np.unique(folds)):
        val_mask   = folds == fold_id
        train_mask = ~val_mask
        fold_save  = str(out_dir / f'fold{fold_id}')
        done_marker = out_dir / f'fold{fold_id}' / 'metric.result'

        logger.info(f'\n--- Fold {fold_id}: {train_mask.sum()} train, {val_mask.sum()} val ---')

        if not done_marker.exists():
            clf = MolTrain(save_path=fold_save, **params)
            clf.fit({'SMILES': [smiles_train[i] for i in np.where(train_mask)[0]],
                     'target': y_train[train_mask].tolist()})
            del clf; gc.collect(); clear_device_cache()

        pred_smi = [smiles_train[i] for i in np.where(val_mask)[0]]
        predictor = MolPredict(load_model=fold_save)
        val_preds = predictor.predict(pred_smi).flatten()
        del predictor; gc.collect(); clear_device_cache()

        oof[val_mask] = val_preds.astype(np.float32)
        logger.info(f'  Fold {fold_id} val MAE = {np.mean(np.abs(val_preds - y_train[val_mask])):.4f}')

    metrics = evaluate_oof_simple(y_train, oof, folds)

    # Final model → test predictions with 10-conformer augmentation
    logger.info('\n--- Final model (all data → test predictions) ---')
    final_save = str(out_dir / 'final')
    clf_final = MolTrain(save_path=final_save, **params)
    clf_final.fit({'SMILES': smiles_train, 'target': y_train.tolist()})

    predictor_final = MolPredict(load_model=final_save)
    aug_smi, aug_idx = augment_smiles(smiles_test, n_aug=10)
    aug_preds = predictor_final.predict(aug_smi).flatten().astype(np.float32)
    test_preds = np.array([
        aug_preds[np.array(aug_idx) == i].mean() for i in range(len(smiles_test))
    ], dtype=np.float32)
    logger.info(f'Test: mean={test_preds.mean():.3f}, std={test_preds.std():.3f}')

    # Save to /content then copy to Drive
    np.save(out_dir / 'oof_predictions.npy', oof)
    np.save(out_dir / 'test_predictions.npy', test_preds)
    with open(out_dir / 'metrics.json', 'w') as fh:
        json.dump(metrics, fh, indent=2)

    drive_out = Path(out_dir_drive)
    drive_out.mkdir(parents=True, exist_ok=True)
    import shutil
    for fname in ['oof_predictions.npy', 'test_predictions.npy', 'metrics.json']:
        shutil.copy(out_dir / fname, drive_out / fname)
    logger.info(f'✅ Saved to Drive: {drive_out}')
    return metrics

print('Training helper ready.')

In [ ]:
# ── Cell 7: Train s4 (LR=5e-4) ───────────────────────────────────────────────
# Expected runtime: ~2–3h on T4
metrics_s4 = train_unimol_variant(
    variant_name='unimol2_s4',
    learning_rate=5e-4,
    out_dir_drive=os.path.join(RESULTS_DIR, 'unimol2_s4')
)
print(f"\ns4 done: MAE={metrics_s4['mae']:.4f}, Spearman={metrics_s4['spearman']:.4f}")

In [ ]:
# ── Cell 8: Train s5 (LR=1e-3, batch=4) ─────────────────────────────────────
# Expected runtime: ~2–3h on T4
# Uses smaller batch for additional diversity (noisier gradients)
_DEVICE_S5 = get_unimol_device_params(base_batch_size=4)

def train_unimol_s5(out_dir_drive):
    """s5 uses batch_size=4 for diversity — override the default batch size."""
    import json
    out_dir = Path('/content/models/unimol2_s5')
    out_dir.mkdir(parents=True, exist_ok=True)
    params = {
        'task': 'regression', 'data_type': 'molecule',
        'model_name': 'unimolv1', 'epochs': 25,
        'learning_rate': 1e-3,
        'batch_size': 4,           # intentionally small for diversity
        'early_stopping': 5,
        'use_amp': _DEVICE_S5['use_amp'],
        'use_gpu': _DEVICE_S5['use_gpu'],
        'remove_hs': False, 'kfold': 1, 'split': 'random',
        'metrics': 'mae', 'conf_cache_level': 0,
    }
    logger.info('\n=== unimol2_s5 (LR=1e-3, batch=4) ===')
    oof = np.zeros(len(y_train), dtype=np.float32)
    for fold_id in sorted(np.unique(folds)):
        val_mask = folds == fold_id; train_mask = ~val_mask
        fold_save = str(out_dir / f'fold{fold_id}')
        done_marker = out_dir / f'fold{fold_id}' / 'metric.result'
        logger.info(f'\n--- Fold {fold_id} ---')
        if not done_marker.exists():
            clf = MolTrain(save_path=fold_save, **params)
            clf.fit({'SMILES': [smiles_train[i] for i in np.where(train_mask)[0]],
                     'target': y_train[train_mask].tolist()})
            del clf; gc.collect(); clear_device_cache()
        predictor = MolPredict(load_model=fold_save)
        val_preds = predictor.predict([smiles_train[i] for i in np.where(val_mask)[0]]).flatten()
        del predictor; gc.collect(); clear_device_cache()
        oof[val_mask] = val_preds.astype(np.float32)
        logger.info(f'  Fold {fold_id} val MAE = {np.mean(np.abs(val_preds - y_train[val_mask])):.4f}')
    metrics = evaluate_oof_simple(y_train, oof, folds)
    final_save = str(out_dir / 'final')
    clf_final = MolTrain(save_path=final_save, **params)
    clf_final.fit({'SMILES': smiles_train, 'target': y_train.tolist()})
    predictor_final = MolPredict(load_model=final_save)
    aug_smi, aug_idx = augment_smiles(smiles_test, n_aug=10)
    aug_preds = predictor_final.predict(aug_smi).flatten().astype(np.float32)
    test_preds = np.array([
        aug_preds[np.array(aug_idx) == i].mean() for i in range(len(smiles_test))
    ], dtype=np.float32)
    np.save(out_dir / 'oof_predictions.npy', oof)
    np.save(out_dir / 'test_predictions.npy', test_preds)
    with open(out_dir / 'metrics.json', 'w') as fh: json.dump(metrics, fh, indent=2)
    import shutil
    drive_out = Path(out_dir_drive); drive_out.mkdir(parents=True, exist_ok=True)
    for fname in ['oof_predictions.npy', 'test_predictions.npy', 'metrics.json']:
        shutil.copy(out_dir / fname, drive_out / fname)
    logger.info(f'✅ Saved to Drive: {drive_out}')
    return metrics

metrics_s5 = train_unimol_s5(
    out_dir_drive=os.path.join(RESULTS_DIR, 'unimol2_s5')
)
print(f"\ns5 done: MAE={metrics_s5['mae']:.4f}, Spearman={metrics_s5['spearman']:.4f}")

In [ ]:
# ── Cell 9: Summary ───────────────────────────────────────────────────────────
print('\n=== TRAINING COMPLETE ===')
print(f"unimol2_s4: MAE={metrics_s4['mae']:.4f}, Spearman={metrics_s4['spearman']:.4f}")
print(f"unimol2_s5: MAE={metrics_s5['mae']:.4f}, Spearman={metrics_s5['spearman']:.4f}")
print(f'\nResults saved to: {RESULTS_DIR}')
print('\nNext steps:')
print('1. Download from Drive: unimol2_s4/ and unimol2_s5/ folders')
print('2. Place in Mac: models/unimol2_s4/ and models/unimol2_s5/')
print('3. Run: python scripts/39_ensemble_phase2.py')